# Project 1: Thermal Sensation — Explain the selected model

This notebook uses permutation importance and SHAP to explore which features contribute to the fitted thermal-sensation model's predictions.

These tools explain the model, not cause and effect. For example, an important temperature feature does not prove that changing temperature will cause a particular sensation.

## 1. Load the fitted Project 1 model and test data

Run Notebooks 01–04 first. The model was selected using training-building cross-validation, then saved after its final test evaluation.

In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import pandas as pd
import shap
from sklearn.inspection import permutation_importance

project_folder = Path.cwd()
if not (project_folder / "data" / "processed").exists():
    project_folder = project_folder.parent

processed_folder = project_folder / "data" / "processed"
model_path = project_folder / "outputs" / "p1_thermal_sensation_model.joblib"

cleaned_data = pd.read_csv(
    processed_folder / "thermal_sensation_cleaned.csv",
    dtype={"building_id": "string", "subject_id": "string"},
    low_memory=False,
)
split_assignments = pd.read_csv(
    processed_folder / "p1_building_split.csv",
    dtype={"building_id": "string"},
)

if not model_path.exists():
    raise FileNotFoundError(
        "Run Notebook 04 first. The fitted model was not found at: "
        f"{model_path}"
    )

best_model = joblib.load(model_path)
model_data = cleaned_data.merge(
    split_assignments,
    on="building_id",
    how="left",
    validate="many_to_one",
)
test_data = model_data.loc[model_data["data_split"] == "test"].copy()

numeric_features = ["ta", "rh", "vel", "clo", "met"]
categorical_features = [
    "season",
    "country",
    "climate",
    "building_type",
    "cooling_type",
]
selected_features = numeric_features + categorical_features
X_test = test_data[selected_features]
y_test = test_data["thermal_sensation"]

print("Test observations for interpretation:", len(test_data))

## 2. Calculate permutation feature importance

Permutation importance measures how much the model's error increases when one input column is shuffled. A larger increase means the model relied more on that feature for these test observations.

We use a reproducible sample of the test data to keep this calculation manageable. The model itself is not retrained during this analysis.

In [ ]:
sample_size = min(2000, len(X_test))
X_importance_sample = X_test.sample(n=sample_size, random_state=42)
y_importance_sample = y_test.loc[X_importance_sample.index]

importance_results = permutation_importance(
    best_model,
    X_importance_sample,
    y_importance_sample,
    scoring="neg_mean_absolute_error",
    n_repeats=5,
    random_state=42,
    n_jobs=1,
)

feature_importance = pd.DataFrame(
    {
        "feature": selected_features,
        "importance_mean": importance_results.importances_mean,
        "importance_std": importance_results.importances_std,
    }
).sort_values("importance_mean", ascending=False)

display(feature_importance)

plt.figure(figsize=(8, 5))
plt.barh(feature_importance["feature"], feature_importance["importance_mean"])
plt.gca().invert_yaxis()
plt.xlabel("Increase in MAE after shuffling")
plt.title("Permutation importance on held-out buildings")
plt.show()

## 3. Use SHAP to inspect model contributions

SHAP explains how encoded input columns move a prediction away from the model's average prediction. Since the model pipeline contains both preprocessing and a regressor, we first transform a small background sample and a small set of test observations using the already-fitted preprocessor.

The background sample helps define the reference prediction. The test sample keeps the explanation focused on observations not used to fit the model.

In [ ]:
background_size = min(100, len(test_data))
explanation_size = min(200, len(test_data))

background_rows = X_test.sample(n=background_size, random_state=1)
explanation_rows = X_test.sample(n=explanation_size, random_state=2)

fitted_preprocessor = best_model.named_steps["prepare_features"]
regression_model = best_model.named_steps["regression"]

# Use the transformations learned during model fitting; do not fit preprocessing again.
background_values = fitted_preprocessor.transform(background_rows)
explanation_values = fitted_preprocessor.transform(explanation_rows)

# Some preprocessing setups return sparse matrices. SHAP works with ordinary arrays here.
if hasattr(background_values, "toarray"):
    background_values = background_values.toarray()

if hasattr(explanation_values, "toarray"):
    explanation_values = explanation_values.toarray()

encoded_feature_names = fitted_preprocessor.get_feature_names_out()

explainer = shap.Explainer(
    regression_model,
    background_values,
    feature_names=encoded_feature_names,
)
shap_values = explainer(explanation_values)

print("SHAP values calculated for", explanation_size, "held-out observations.")

In [ ]:
# The bar plot summarizes average absolute contributions across observations.
shap.plots.bar(shap_values)

# The beeswarm plot also shows whether feature values move predictions up or down.
shap.plots.beeswarm(shap_values)

## Interpretation reminders

- Permutation importance and SHAP describe how this fitted model uses its inputs; they do not prove physical causation.
- Correlated measurements can share or mask importance. Treat rankings as clues to investigate, not definitive engineering laws.
- The one-hot encoded country, climate, and other categories appear as separate SHAP features. Their contributions can be grouped for a simpler report if needed.
- These results explain thermal-sensation predictions only. They are not measured energy savings or thermostat-control instructions.